## Exploring META data and Generating Subfolders for FB_INF project
### Targeted Modulation Version (GenV6)

#### LOADING THE NECESSARY LIBRARIES TO EXPLORE THE METADATA GENERATED

In [1]:

%load_ext autoreload
%autoreload 2

from wormholes import *
from wormholes.perturb import *
from wormholes.tools.triplets_vis_tools import *
import matplotlib.pyplot as plt
import matplotlib.image as img
from argparse import Namespace
import os
import xarray as xr #this is to be able to read the .nc type of file
import shutil
import random 
import pandas as pd

In [2]:
# Set the HOME environment variable to your current working directory
os.environ["HOME"] = "/project/3018078.01/Gaziv/Wormholes_FB"

# Now the `os.path.expanduser("~")` will resolve to this directory
print(os.path.expanduser("~")) 

/project/3018078.01/Gaziv/Wormholes_FB


This next part of the script is loading the meta data file and does some exploration of the variables that have been defined there. 

In [3]:
#this is the directory where they will find the images (change as you deem necessary)

exp_root = f"{PROJECT_ROOT}/results/cache/gen_v6"

# Load the metadata file
metadata = xr.open_dataset(f"{exp_root}/meta.nc")

# Print metadata structure
# print(metadata)

# List variables in the dataset
print(metadata.variables)

Frozen({'model_name': <xarray.Variable (image_id: 46200)>
array(['resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0', ...,
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0',
       'resnet50_robust_mapped_RIN_l2_3_0'], dtype=object), 'budget': <xarray.Variable (image_id: 46200)>
array([ 0.,  0.,  0., ..., 30., 30., 30.]), 'n_iter': <xarray.Variable (image_id: 46200)>
array([   0,    0,    0, ..., 2000, 2000, 2000]), 'step_size': <xarray.Variable (image_id: 46200)>
array([0., 0., 0., ..., 2., 2., 2.]), 'interp_alpha': <xarray.Variable (image_id: 46200)>
array([nan, nan, nan, ..., nan, nan, nan]), 'target_class_name': <xarray.Variable (image_id: 46200)>
array(['turtle', 'lizard', 'bird', ..., 'insect', 'rabbit', 'gazelle'],
      dtype=object), 'orig_class_name': <xarray.Variable (image_id: 46200)>
array(['OOD-frog', 'OOD-frog', 'OOD-frog', ..., 'OOD-primate', 'OOD-primate',
     

I am now transforming the .nc file into a Pandas dataframe, as this is the type of structure I am familiar with and allows me to do the selection of relevant information I need for my experiment

In [4]:
#Depending on the size of the meta.nc file. The kernel might get overwhemled when trying to trasnform it to a pandas dataframe. 
#Therefore, the next couple of code lines are to only pass the information that is relevant (reducing the size of df)
selected_vars = ["model_name", "budget", "image_id",'class_index', 'pred_logit',
                 'model_subject_name', 'orig_class_name', 'orig_name', 'target_class_name']
df = metadata[selected_vars].to_dataframe().reset_index()

# Convert the dataset to a Pandas DataFrame (for the whole df, use the following line of code)
# df = metadata.to_dataframe().reset_index()

# Inspect the metadata structure
print(df.head())

# Inspect the columns in the Pandas Dataframe
print(df.columns)

#information about the data in the columns
df.info(verbose=False)

                               image_id  class_index  \
0  65c445b6a56020704234dcb9ed34f4b3.png            0   
1  65c445b6a56020704234dcb9ed34f4b3.png            0   
2  65c445b6a56020704234dcb9ed34f4b3.png            0   
3  65c445b6a56020704234dcb9ed34f4b3.png            0   
4  65c445b6a56020704234dcb9ed34f4b3.png            0   

                      model_subject_name                         model_name  \
0  resnet50_robust_mapped_RIN_l2_10_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
1   resnet50_robust_mapped_RIN_l2_1_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
2      resnet50_robust_mapped_RIN_l2_3_0  resnet50_robust_mapped_RIN_l2_3_0   
3   resnet50_robust_mapped_RIN_l2_3_0_v2  resnet50_robust_mapped_RIN_l2_3_0   
4         resnet50_vanilla_mapped_RIN_v2  resnet50_robust_mapped_RIN_l2_3_0   

   budget  pred_logit orig_class_name                 orig_name  \
0     0.0    2.090741        OOD-frog  OOD/frog/frog_00022.JPEG   
1     0.0    2.415928        OOD-frog  OOD/frog/frog_0

I am now filtering the dataframe so it only includes the parameters that are relevant for my experiment. 
With the filtered data frame I can then create folders with the relevant images for the experiment (with the correct names)

NOTE: model_name is the variable used to define which model is being used to generate the adversarial images. model_subject_name is all the other models that will be tested on the adversarial images generated by model_name and compare their performances. 

In [5]:
#VISUALIZATIONS OF THE WHOLE DATAFRAME

# print(df['model_name'].unique()) # displays all the networks available to extract images from
# print(df['model_subject_name'].unique()) #the v2 versions of the networks are just initialized using different seeds
# print(df['budget'].unique()) #all the budget options I can select images from


# Apply filtering (as model subject name makes a double entry on all meta data -- reducing the size)
filtered_df = df[(df['model_subject_name'] == 'resnet50_robust_mapped_RIN_l2_3_0')] 


# MORE VISUALIZATIONS
# print(filtered_df.head())
# print(filtered_df.columns)
# print(filtered_df['model_name'])
# print(len(filtered_df['image_id'].unique()))
# filtered_df.shape
# print(filtered_df['class_index'].unique())


Now, I want to generate code that creates a new folder with all my perturbed images. 
I will devise a specific form of naming them that will make it easier for me to use in the actual experiment. 

Naming convention = orig(origclass)_target(targetclass)_budget_imgnumber

I will create code to generate a folder that includes the images for a specific budget. 

As we have not decided if we are going with 10 or with 12 classes, I will create either two functions or one function with a parameter choice to decide to create a dataset that already satisfies the needs of the experiment.


In [28]:
#Creating a new folder with images per class, renamed and (nearly) ready for me to use for the experiments 

def stimulus_set_folder_per_budget(class_num, df, budget, output_folder_root="/project/3018078.01/FB_INF/stimulus_set", seed=0):
    """
    Creates a stimulus set by selecting images per class based on budget constraints.
    The selected images are stored in a designated folder. It selects a specific amount of images
    so that it matches with the needs from the FB_INF project.
    
    Parameters:
        class_num (int): The number of classes (10 or 12) that dictate selection rules.
        df (pd.DataFrame): The dataframe containing image metadata.
        budget (float): The budget constraint for filtering the dataframe.
        output_folder_root (str): Root directory for storing selected images.
        seed (int): Random seed for reproducibility.
    
    Returns:
        pd.DataFrame: The final selected images dataframe.
        The folder containing all the images with the naming convention (orig-'class_name'_target-'class_name'_'budget'_'i'.png")
        established in the output directory. 
        
    """

    # Set up output folder for the selected stimulus set
    folder_dir_path = os.path.join(output_folder_root, f"{budget}budget_{class_num}classes")
    # Create the folder if it doesn't exist
    os.makedirs(folder_dir_path, exist_ok=True)

    #Apply filter so it only includes the desired budget and the number of classes requested (with its corresponding number of images)
    if class_num == 10:
        df = df[(df['budget'] == budget) & (~df["orig_class_name"].isin(["OOD-cat", "OOD-gazelle"])) 
        & (~df["target_class_name"].isin(["cat", "gazelle"]))]

        samples_per_class = 36
        targets_per_sample = 4
    
    elif class_num == 12:
        df = df[(df['budget'] == budget)] 
        samples_per_class = 22
        targets_per_sample = 2

    else:
        raise ValueError(f"Argument class_num must be one of the two options: 10 or 12.")
    

    # Ensure randomization is consistent
    random.seed(seed) 

    # PART1: Select a fixed number of unique images per orig_class_name
    
    selected_rows = [] # Initialize list to store selected rows
    # Loop over each original class to select only a certain amount of images
    for _, class_df in df.groupby("orig_class_name"):
        unique_images = class_df["orig_name"].unique()

        # Ensure enough images exist for sampling
        if len(unique_images) < samples_per_class:
            raise ValueError(f"Class {class_df['orig_class_name'].iloc[0]} has fewer than 22 images available.")

        # Randomly select unique images
        selected_origimage_ids = random.sample(list(unique_images), samples_per_class)  # Select number unique images
        selected_rows.append(class_df[class_df["orig_name"].isin(selected_origimage_ids)])

    # Combine all selected rows into a new dataframe
    selected_df = pd.concat(selected_rows, ignore_index=True)

    
    # PART2: Ensure unique image_ids across target classes (so there is a balance on the target options per class and no repeated origin images)
    selected_target_orig_pair_rows =[] # Initialize list to store selected rows

    
    for orig_class_name, orig_class_df in selected_df.groupby("orig_class_name"):
        final_selected_rows = []
        selected_orig_name = set()
        for target_class_name, target_df in orig_class_df.groupby("target_class_name"):
        
            # Remove images that are already selected
            target_df = target_df[~target_df["orig_name"].isin(selected_orig_name)]
            orig_class_df = orig_class_df[~orig_class_df["orig_name"].isin(selected_orig_name)]

            # Ensure we have at least unique images for this target class
            if len(target_df) < targets_per_sample:
                raise ValueError(f"Target class '{target_class_name}' in orig_class '{orig_class_name}' has fewer than number unique distractor images.")
            
            # Randomly select number unique distractor images
            # selected_target_images = target_df.sample(n=targets_per_sample, random_state=seed)
            selected_target_images = target_df.drop_duplicates(subset="orig_name").sample(n=targets_per_sample, random_state=seed) #why drop duplicates? because there are many duplicated rows in dataframe for performance comparison in networks. 

            
            # Add selected image_ids to the set to prevent reuse
            selected_orig_name.update(selected_target_images["orig_name"])
            final_selected_rows.append(selected_target_images)

        # Combine selected images for this orig_class_name
        selected_target_orig_pair_rows.append(pd.concat(final_selected_rows, ignore_index=True))

    # Combine all selected rows into a new dataframe
    final_df = pd.concat(selected_target_orig_pair_rows, ignore_index=True)

    #PART3: Copy all the selected images to the output_folder_directory using the designated name convention
    for orig_class_name, class_df in final_df.groupby("orig_class_name"):
        for i, row in enumerate(class_df.itertuples(index=False)):
            file_name = f"orig-{row.orig_class_name}_target-{row.target_class_name}_{row.budget}_{i+1}.png"
            destination_path = os.path.join(folder_dir_path, file_name)
            source_path = os.path.join(exp_root, f'images/{row.image_id}')
            
            # Ensure the file exists before copying
            try:
                shutil.copy(source_path, destination_path)
            except FileNotFoundError:
                print(f"Warning: File {source_path} not found, skipping.")


    return final_df
        

budget_7_5 = stimulus_set_folder_per_budget(class_num = 10, df = filtered_df, budget = 0, seed=77)


# Check if it worked:

# print(tryout["orig_class_name"].value_counts())  # Should be 36 per orig_class
# print(tryout.groupby(["orig_class_name", "target_class_name"])["orig_name"].nunique())  # Should be 4 per target_class
# print(tryout["orig_name"].nunique())  # Should not have duplicates! 
    

IMAGES FLAGGED AS UNUSABLE FROM 10 CLASSES, SEED 1

Two animals in the same image:

- Bear 10, 28
- Bird 3, 6, 26
- Crab 2, 7, 10, 16, 32
- Dog 2, 21
- Frog 4, 6, 13
- Insect 27, 28
- Lizard 3, 4, 12, 31, 34
- Monkey 18, 19 
- Rabbit 14, 28, 31, 36
- Turtle 1, 3, 16


Cropped:
- Bear 23!
- Crab 7! 
- Dog 6!
- Frog 33! 
- Lizard 7!, 17!

dog 32

This next section is an exploration of the dataset to replace the cropped images that are unusable for the experiment 

In [13]:
def extra_image_replacement(df_original, df_perturbed, orig_class, target_class, budget, num_replacements, output_folder="/project/3018078.01/FB_INF/stimulus_set", total_class_num =10, seed = 1):
    """
    Selects extra replacement images from `df_original`, ensuring they are not already present in `df_perturbed`. 
    The selected images are then copied into a designated output directory. This is to replace images that were deemed unusable in previous manual inspection.

    Parameters:
    -----------
    df_original : pd.DataFrame
        The original dataset containing all available images, their class labels, target class labels, and budget information.
    df_perturbed : pd.DataFrame
        A dataset containing images that have already been selected for the FB_INF experiment.
    orig_class : str
        The original class name of the images to be selected.
    target_class : str
        The target class name associated with the selected images.
    budget : int or float
        The budget category used to filter the dataset.
    num_replacements : int
        The number of replacement images to select.
    output_folder : str, default="/project/3018078.01/FB_INF/stimulus_set"
        The root directory where the selected images will be stored.
    total_class_num : int, default=10
        The total number of classes in the dataset, used for naming the output folder.

    )

    Returns: 
    ------------

    The pd.Dataframe containin the information for the extra images. 
    It adds the images for the parameters inserted in the folder with the other perturbed images


    """
    # Ensure randomization is consistent
    random.seed(seed)
    
    # Set up output folder for the selected stimulus set
    folder_dir_path = os.path.join(output_folder, f"{budget}budget_{total_class_num}classes")

    # Filter by original class, target class, and budget
    explore_images_df = df_original[
        (df_original['orig_class_name'] == orig_class) & 
        (df_original["target_class_name"] == target_class) & 
        (df_original["budget"] == budget) & 
        (df_original["class_index"] == 1)
    ]

    # Checking which images are not already selected when generating the data
    unique_to_orig_df = set(explore_images_df["orig_name"].unique())
    unique_to_orig_df.difference_update(df_perturbed["orig_name"].unique())  

    # Ensure we have enough images to sample from
    if len(unique_to_orig_df) < num_replacements:
        raise ValueError(f"Not enough unique replacement images for '{orig_class}' targeting '{target_class}' (Available: {len(unique_to_orig_df)}, Needed: {num_replacements}).")


    # Randomly select `num_replacements` unique images
    selected_origimage_ids = random.sample(sorted(unique_to_orig_df), num_replacements)  

    # Filter `explore_images_df` to keep only the selected images
    explore_images_df = explore_images_df[explore_images_df["orig_name"].isin(selected_origimage_ids)]

    print(explore_images_df.head())

    #return explore_images_df  # Return the filtered DataFrame

    for i, row in enumerate(explore_images_df.itertuples(index=False)):
            file_name = f"orig-{row.orig_class_name}_target-{row.target_class_name}_{row.budget}_{i+1}_EXTRA.png"
            destination_path = os.path.join(folder_dir_path, file_name)
            source_path = os.path.join(exp_root, f'images/{row.image_id}')
            
            # Ensure the file exists before copying
            try:
                shutil.copy(source_path, destination_path)
            except FileNotFoundError:
                print(f"Warning: File {source_path} not found, skipping.")

    

extra_image_replacement(df_original = filtered_df, df_perturbed = budget_7_5, orig_class = 'OOD-crab', target_class = 'lizard', budget = 0, num_replacements = 1, seed = 5)
 







                                    image_id  class_index  \
154567  e81874ac37afb3c7361929c16a9b8eb5.png            1   

                       model_subject_name                         model_name  \
154567  resnet50_robust_mapped_RIN_l2_3_0  resnet50_robust_mapped_RIN_l2_3_0   

        budget  pred_logit orig_class_name                 orig_name  \
154567     0.0    1.985605        OOD-crab  OOD/crab/crab_00029.JPEG   

       target_class_name  
154567            lizard  


In order for the folders for all budgets to be the same, with the same names and the same amount of images I will be doing some cleaning up of the old images.

In [162]:

def move_flagged_images(budget):
    """
    Moves all files containing '_OLD' in their filename from a specific budget folder
    to a designated flagged images folder.

    Args:
        budget (str): The budget identifier used to construct the folder path.
    """

    # Define the source and target directories
    origin_folder = '/project/3018078.01/FB_INF/stimulus_set'
    budget_folder_old = os.path.join(origin_folder, f'{budget}budget_10classes')
    target_folder_old = os.path.join(origin_folder, 'flagged_images')

    # Ensure target directory exists; if not, create it
    os.makedirs(target_folder_old, exist_ok=True)

    # Iterate over all files in the source directory
    for filename in os.listdir(budget_folder_old):
        # Check if '_OLD_' appears in the filename (case-sensitive)
        if '_OLD' in filename:
            # Construct full source and destination paths
            source_path = os.path.join(budget_folder_old, filename)
            destination_path = os.path.join(target_folder_old, filename)

            # Move the file
            shutil.move(source_path, destination_path)
            print(f"Moved: {filename} -> {destination_path}")



budgets = [0, 7.5, 10, 12.5, 15]

for budget in budgets:
    move_flagged_images(budget)


Moved: orig-OOD-bird_target-crab_0.0_6_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-bird_target-crab_0.0_6_OLD.png
Moved: orig-OOD-bird_target-frog_0.0_16_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-bird_target-frog_0.0_16_OLD.png
Moved: orig-OOD-crab_target-bear_0.0_2_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-crab_target-bear_0.0_2_OLD.png
Moved: orig-OOD-dog_target-rabbit_0.0_32_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-dog_target-rabbit_0.0_32_OLD.png
Moved: orig-OOD-bird_target-crab_7.5_6_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-bird_target-crab_7.5_6_OLD.png
Moved: orig-OOD-bird_target-frog_7.5_16_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-bird_target-frog_7.5_16_OLD.png
Moved: orig-OOD-crab_target-bear_7.5_2_OLD.png -> /project/3018078.01/FB_INF/stimulus_set/flagged_images/orig-OOD-crab_tar

The following function is checking that the same file names and count of files are present in all the folders. This to ensure that there are no mistakes happening while cleaning up the data.

In [163]:

origin_folder = "/project/3018078.01/FB_INF/stimulus_set"

def get_standardized_filenames(folder):
    """Returns a mapping of original filenames to standardized filenames ignoring the budget number."""
    filename_map = {}  # Dictionary to store {original_filename: standardized_filename}
    
    pattern = re.compile(r"_\d+(\.\d+)?_")  # Matches _10_, _7.5_, _12.5_, etc.

    for file in os.listdir(folder):
        standardized_name = pattern.sub("_BUDGETPLACEHOLDER_", file, count=1)  # Replace only the budget part
        filename_map[file] = standardized_name  # Store the mapping of original to standardized name
    
    return filename_map

# Get all budget folders inside origin_folder
budget_folders = [os.path.join(origin_folder, f) for f in os.listdir(origin_folder) 
                  if os.path.isdir(os.path.join(origin_folder, f))]

if not budget_folders:
    print("No budget folders found.")
else:
    reference_folder = budget_folders[0]
    reference_filenames = get_standardized_filenames(reference_folder)
    reference_set = set(reference_filenames.values())  # Set of standardized filenames
    reference_count = len(reference_set)

    print(f"Using {reference_folder} as reference.")

    for folder in budget_folders[1:]:
        current_filenames = get_standardized_filenames(folder)
        current_set = set(current_filenames.values())

        # Compare number of files
        if len(current_set) != reference_count:
            print(f"Mismatch in file count for folder: {folder} (Expected: {reference_count}, Found: {len(current_set)})")

        # Compare file names
        if current_set != reference_set:
            missing_files = reference_set - current_set  # Files in reference but missing in this folder
            extra_files = current_set - reference_set  # Files in this folder but not in reference

            print(f"Mismatch in file names for folder: {folder}")
            
            if missing_files:
                print(f"  Missing files:")
                for missing in missing_files:
                    # Find the original filenames from the reference folder
                    original_name = [name for name, std_name in reference_filenames.items() if std_name == missing]
                    print(f"    - {original_name[0]} (Expected a version in this folder)")

            if extra_files:
                print(f"  Extra files:")
                for extra in extra_files:
                    # Find the original filenames in the current folder
                    original_name = [name for name, std_name in current_filenames.items() if std_name == extra]
                    print(f"    - {original_name[0]} (Unexpected file in this folder)")

    print("Check complete.")


Using /project/3018078.01/FB_INF/stimulus_set/12.5budget_10classes as reference.
Check complete.
